<a href="https://colab.research.google.com/github/Imran1hp/Deep-Learning-/blob/main/PyTorch_Computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor
import  matplotlib.pyplot as plt




## Download the datastes from torchvision

In [ ]:
from torchvision import datasets

train_data = datasets.FashionMNIST(
    root = 'data', # where to download the data to
    train = True, # do we want the train data set
    download = True,# do we want to downlaod it
    transform = torchvision.transforms.ToTensor(), # do we want to transform the data into tensor
    target_transform = None # to we wnat to change the target labels

)

test_data = datasets.FashionMNIST(
    root ='data',
    train = False,
    download = True,
    transform = ToTensor(),
    target_transform = None

)

In [ ]:
len(train_data),len(test_data)

In [ ]:
image,labels = train_data[0]

In [ ]:
image ,labels

In [ ]:
class_name = train_data.classes

In [ ]:
class_name

In [ ]:
class_to_idx = train_data.class_to_idx
class_to_idx

In [ ]:
train_data.targets

In [ ]:
image.shape ,labels

In [ ]:
import matplotlib.pyplot as plt

image , labels = train_data[0]
print(f"shape of the image {image.shape}")
plt.title(labels)
plt.imshow(image.squeeze())

In [ ]:
image.shape

In [ ]:
image.squeeze().shape

In [ ]:
plt.imshow(image.squeeze() , cmap = 'gray')
plt.title(class_name[labels])
plt.axis(False)

In [ ]:
torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))
row , col = 4,4

for i in range(1, row * col + 1 ):
  random_idx = torch.randint( 0 , len(train_data), size = [1]).item()
  image , labels = train_data[random_idx]
  fig.add_subplot(row , col ,i)
  plt.imshow(image.squeeze(),cmap='gray')
  plt.title(class_name[labels])
  plt.axis(False)


## Turn our data in to mini batches so our machine ram or memory don't consume by it fully default batch size is 32  

using DataLoader function

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_data_loader = DataLoader(train_data , batch_size = BATCH_SIZE , shuffle = True)

test_data_loader = DataLoader(test_data , batch_size = BATCH_SIZE , shuffle = False)

train_data_loader has 1875  batch inside it and each batch has 32 images  

In [ ]:
print(f"Length of the train data loader: {len(train_data_loader)} of each batch size {BATCH_SIZE}")
print(f"Length of the test data loader: {len(test_data_loader)} of each batch size {BATCH_SIZE}")

In [ ]:
60000/32

In [ ]:
10000/32

In [ ]:
train_feature_batch , train_label_batch = next(iter(train_data_loader))
train_feature_batch.shape , train_label_batch.shape

 32 is batch size 1 is color channel 28 and 28 hight and width

 ## train_feature_batch is 1 batch which is we iterating one by one by next(iter()) function  from train_data_loader

In [ ]:
len(train_feature_batch)

In [ ]:
torch.manual_seed(42)

random_idx = torch.randint( 0 , len(train_feature_batch) , size=[1]).item()
img , labels = train_feature_batch[random_idx] , train_label_batch[random_idx]

plt.imshow(img.squeeze() , cmap = 'gray')
plt.title ( class_name[labels])
plt.axis(False)
print(f"Image size {img.shape}")
print(f"Lable: {labels} lable shape {labels.shape}")

 ## Flatten the input by nn.flatten module

In [ ]:
from torch import nn

flatten_output = nn.Flatten()
x = train_feature_batch[0]
output = flatten_output(x)
print(f" Original shape of the image: {x.shape}")
print(f" Flatten shape of the image: {output.shape}")

In [ ]:
device = ""
if torch.cuda.is_available:
  device = "cuda"
else:
  device = "cpu"

In [ ]:
class FashionMNISTModelv0(nn.Module):
  def __init__(self , input_shape =int, hidden_unit = int , output_shape = int):
    super().__init__()
    self.layer_stack = nn.Sequential(
      nn.Flatten(),
      nn.Linear(in_features = input_shape , out_features = hidden_unit),
      nn.Linear(in_features = hidden_unit , out_features = output_shape)
    )
  def forward(self , x):
    return self.layer_stack(x)

In [ ]:
class_name

In [ ]:
torch.manual_seed(42)
model_0 = FashionMNISTModelv0(input_shape = 28*28 , hidden_unit = 10 , output_shape = len(class_name)).to(device)

In [ ]:
model_0

In [ ]:
dummy_x = torch.rand([1,1 ,28,28]).to(device)
model_0(dummy_x).shape

In [ ]:
def accuarcy_fn ( y_true , y_preds ):
  correct = torch.eq(y_true , y_preds).sum().item()
  acc = (correct/len(y_preds))*100
  return acc

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_0.parameters() , lr =0.1)

In [ ]:
from timeit import default_timer as timer

def print_train_time (start: float , end: float , device: torch.device = None):
  total_time = end - start
  print(f"Train time on {device} is {total_time:.3f}s")
  return total_time

In [ ]:
start_time = timer()
cal = 10**2 * 8**3

end_time = timer()
print_train_time(start=start_time , end=end_time , device = "cpu")

## Train a model in batch of data

In [ ]:
len(train_data_loader)

In [ ]:
from tqdm.auto import tqdm

torch.manual_seed(42)
torch.cuda.manual_seed(42)
starting_time = timer()

epochs =3

for epoch in tqdm(range(epochs)):
      print(f"Epochs:{epoch}\n-----")
      train_loss = 0

      for batch ,(X,y ) in enumerate(train_data_loader):
        X, y = X.to(device), y.to(device) # Move data to target device

        model_0.train()

        y_preds = model_0(X)

        loss =loss_fn(y_preds , y)
        train_loss +=loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        if batch %400 == 0:
         print(f"Look at {batch*len(X)} / {len(train_data_loader.dataset)} sample")

      train_loss /=len(train_data_loader) # Average the loss over all batches

      test_loss , test_acc = 0 ,0
      model_0.eval()

      with torch.inference_mode():

        for test_X , test_y in test_data_loader:
          test_X, test_y = test_X.to(device), test_y.to(device) # Move data to target device
          test_pred = model_0(test_X)

          test_loss += loss_fn(test_pred , test_y) # Sum test loss

          test_acc += accuarcy_fn(y_true = test_y , y_preds = test_pred.argmax(dim =1)) # Sum test accuracy

        test_loss /=len(test_data_loader) # Average test loss
        test_acc /= len(test_data_loader) # Average test accuracy
      print(f"Train_loss: {train_loss:.4f} |  Test loss: {test_loss:.4f} | Test_ACC: {test_acc:.4f}")


end_time = timer() # Fix: Assign timer() to end_time

print_train_time(start_time , end_time , device =str(next(model_0.parameters()).device)) # Fix: Pass correct device

In [ ]:
torch.manual_seed(42)
def model_eval(model: torch.nn.Module , data_loader: torch.utils.data.DataLoader , loss_fn : torch.nn.Module , accuarcy_fn):
  loss , acc = 0, 0
  with torch.inference_mode():
    for X, y in data_loader:
      X , y = X.to(device) ,y.to(device)

      y_preds = model(X)

      loss = loss_fn(y_preds , y)
      loss+=loss

      acc += accuarcy_fn(y_true = y , y_preds = y_preds.argmax(dim=1))


    loss/=len(data_loader)
    acc/=len(data_loader)

  return {"Model_name": model.__class__.__name__,
          "Model_loss": loss.item(),
          "Model_Acc": acc}





In [ ]:
model_0_result = model_eval( model = model_0 , data_loader = test_data_loader ,
                            loss_fn = loss_fn , accuarcy_fn = accuarcy_fn)


In [ ]:

model_0_result

## Create a non liner Model with ReLU activation functin

In [ ]:
class FashionMNISTModelV1(nn.Module):
  def __init__(self , input_features: int , hidden_units:int , output_features: int):
    super().__init__()
    self.layer_stack = nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features = input_features , out_features = hidden_units),
        nn.ReLU(),
        nn.Linear(in_features = hidden_units , out_features = output_features ),
        nn.ReLU()
    )
  def forward(self , x:torch.tensor):
    return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)
model_1 = FashionMNISTModelV1(input_features = 28* 28 , hidden_units = 10 , output_features = len(class_name)).to(device)

In [ ]:
next(model_1.parameters()).device

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_1.parameters(), lr =0.1)

In [ ]:
def train_step(model: torch.nn.Module ,dataloader:torch.utils.data.DataLoader , loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer  ,accuarcy_fn , device:torch.device = device ):

  train_loss , train_acc = 0 , 0
  model.train()

  for batch ,(X,y) in enumerate(dataloader):

    X ,y = X.to(device ) , y.to(device)

    y_preds = model(X)

    loss = loss_fn(y_preds , y)
    train_loss += loss
    train_acc += accuarcy_fn( y_true = y ,y_preds = y_preds.argmax(dim=1) )

    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

  train_loss/=len(dataloader)
  train_acc /=len(dataloader)

  print(f"Train Loss: { train_loss:.5f}  | Train_acc: { train_acc:.2f}")


In [ ]:
def test_step (model: torch.nn.Module , dataloader: torch.utils.data.DataLoader ,
               loss_fn: torch.nn.Module ,
               accuarcy_fn , device: torch.device = device):
  model.eval()
  with torch.inference_mode():
    test_loss , test_acc  = 0,0
    for  X, y in dataloader :
      X ,y = X.to(device) ,y.to(device)

      y_preds = model(X)

      loss = loss_fn (y_preds  , y)
      test_loss += loss

      acc = accuarcy_fn(y_true = y , y_preds = y_preds.argmax(dim = 1))
      test_acc += acc

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    print(f" Test loss: {test_loss:.5f}  |  Test Acc: {test_acc:.2f}")






In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model_1.parameters() ,lr = 0.1)

In [ ]:

torch.manual_seed(42)
torch.cuda.manual_seed(42)
start_timer = timer()
epochs = 4
for epoch in tqdm(range(epochs)):
  print(f"Epoch : {epoch}\n------------")
  train_step(model=model_1 , dataloader = train_data_loader, loss_fn = loss_fn , accuarcy_fn=accuarcy_fn , optimizer = optimizer )
  test_step( model_1 , dataloader = test_data_loader , loss_fn =nn.CrossEntropyLoss(), accuarcy_fn = accuarcy_fn)




end_timer = timer()
total_train_time = print_train_time(start = start_timer , end=end_timer , device = device)


In [ ]:
model_0_result

In [ ]:
from google.colab import output
output.clear()
